### Pin classification with zero-shot CLIP ViT-L/14


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from peft import LoraConfig, TaskType, get_peft_model

BACKEND = 'clip'
MODEL_ID = 'openai/clip-vit-large-patch14'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "pin_classification"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "places365_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "food_101_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "inaturalist_pin_manifest.parquet",
]

if BACKEND == "clip":
    from transformers import CLIPModel, CLIPProcessor
    model = CLIPModel.from_pretrained(MODEL_ID)
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
else:
    from transformers import AutoModel, AutoProcessor
    model = AutoModel.from_pretrained(MODEL_ID)
    processor = AutoProcessor.from_pretrained(MODEL_ID)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
sns.set_theme(style="whitegrid")


In [ ]:
def load_pin_frame() -> pd.DataFrame:
    for candidate in PROCESSED_CANDIDATES:
        if candidate.exists():
            frame = pd.read_parquet(candidate)
            frame["source_dataset"] = candidate.stem.replace("_pin_manifest", "")
            return frame
    raise FileNotFoundError("Run one of the pin dataset notebooks first to create a processed pin manifest.")


pin_df = load_pin_frame()
pin_df["image_paths"] = pin_df["image_paths_json"].map(json.loads)
pin_df["text_bundle"] = (
    pin_df["title"].fillna("") + " [SEP] " +
    pin_df["description"].fillna("") + " [SEP] " +
    pin_df["board_name"].fillna("") + " [SEP] " +
    pin_df["tags"].fillna("")
)
classes = sorted(pin_df["label_name"].unique().tolist())

train_df, valid_df = train_test_split(
    pin_df,
    test_size=0.2 if len(pin_df) >= 100 else 0.3,
    stratify=pin_df["label_name"] if pin_df["label_name"].nunique() > 1 else None,
    random_state=42,
)
print("Dataset:", pin_df["source_dataset"].iloc[0], "| pins:", len(pin_df), "| classes:", len(classes))


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


prompt_templates = [
    "a photo for a pin about {label}",
    "a pinterest pin related to {label}",
    "an image collection representing {label}",
]


@torch.inference_mode()
def encode_images(images: list[Image.Image]) -> np.ndarray:
    if BACKEND == "clip":
        inputs = processor(images=images, return_tensors="pt").to(device)
        return model.get_image_features(**inputs).detach().cpu().numpy()
    inputs = processor(images=images, return_tensors="pt").to(device)
    return model.get_image_features(**inputs).detach().cpu().numpy()


@torch.inference_mode()
def encode_texts(texts: list[str]) -> np.ndarray:
    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
    if BACKEND == "clip":
        return model.get_text_features(**inputs).detach().cpu().numpy()
    return model.get_text_features(**inputs).detach().cpu().numpy()


class_prompts = [prompt_templates[0].format(label=label) for label in classes]
class_text_emb = encode_texts(class_prompts)
class_text_emb = class_text_emb / np.linalg.norm(class_text_emb, axis=1, keepdims=True)


def pin_zero_shot_proba(row: pd.Series) -> np.ndarray:
    images = [Image.open(path).convert("RGB") for path in row["image_paths"]]
    image_emb = encode_images(images)
    image_emb = image_emb / np.linalg.norm(image_emb, axis=1, keepdims=True)
    image_scores = image_emb @ class_text_emb.T
    img_vote = image_scores.mean(axis=0)

    prompt_variants = [template.format(label=row["label_name"]) for template in prompt_templates]
    _ = prompt_variants
    if BACKEND == "clip":
        text_emb = encode_texts([row["text_bundle"]])
    else:
        text_emb = encode_texts([row["text_bundle"]])
    text_emb = text_emb / np.linalg.norm(text_emb, axis=1, keepdims=True)
    txt_vote = (text_emb @ class_text_emb.T)[0]

    logits = 0.75 * img_vote + 0.25 * txt_vote
    exp = np.exp(logits - logits.max())
    return exp / exp.sum()


zero_shot_probas = np.vstack(valid_df.apply(pin_zero_shot_proba, axis=1).tolist())
metrics = compute_multiclass_metrics(valid_df["label_name"], zero_shot_probas, classes)
metrics


In [ ]:
sample_images = [Image.open(path).convert("RGB") for path in valid_df.iloc[0]["image_paths"]]
if BACKEND == "clip":
    inspect_inputs = processor(text=class_prompts[:8], images=sample_images[:2], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inspect_inputs, output_hidden_states=True)
    image_emb = outputs.image_embeds.detach().cpu().numpy()
else:
    inspect_inputs = processor(text=class_prompts[:8], images=sample_images[:2], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = model(**inspect_inputs, output_hidden_states=True)
    image_emb = outputs.image_embeds.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(pd.DataFrame(image_emb).corr(), cmap="mako", ax=axes[0])
axes[0].set_title("Image embedding correlation")
module_overview = pd.DataFrame(
    [
        {"name": name, "module": module.__class__.__name__}
        for name, module in list(model.named_modules())[:80]
    ]
)
sns.barplot(data=module_overview["module"].value_counts().head(10).reset_index(name="count"), x="count", y="module", ax=axes[1])
axes[1].set_title("Top module types")
plt.tight_layout()


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


def aggregate_pin_embedding(row: pd.Series) -> np.ndarray:
    images = [Image.open(path).convert("RGB") for path in row["image_paths"]]
    image_emb = encode_images(images)
    image_emb = image_emb.mean(axis=0)
    text_emb = encode_texts([row["text_bundle"]])[0]
    return np.concatenate([image_emb, text_emb], axis=0)


train_x = np.vstack(train_df.apply(aggregate_pin_embedding, axis=1).tolist())
valid_x = np.vstack(valid_df.apply(aggregate_pin_embedding, axis=1).tolist())
clf = LogisticRegression(max_iter=4000, multi_class="multinomial")
clf.fit(train_x, train_df["label_name"])
valid_probas = clf.predict_proba(valid_x)
probe_metrics = compute_multiclass_metrics(valid_df["label_name"], valid_probas, clf.classes_)

run_dir = ARTIFACT_ROOT / ("pin_" + BACKEND + "_zero_shot")
run_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame(valid_probas, columns=clf.classes_).to_parquet(run_dir / "validation_probas.parquet", index=False)
(run_dir / "metrics.json").write_text(json.dumps(probe_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
probe_metrics


In [ ]:
target_modules = ["q_proj", "k_proj", "v_proj", "out_proj"] if BACKEND == "clip" else ["q_proj", "k_proj", "v_proj", "out_proj"]
lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=target_modules,
    use_dora=False,
)
dora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=target_modules,
    use_dora=True,
)

lora_model = get_peft_model(model.__class__.from_pretrained(MODEL_ID), lora_cfg)
dora_model = get_peft_model(model.__class__.from_pretrained(MODEL_ID), dora_cfg)
lora_model.save_pretrained(ARTIFACT_ROOT / ("pin_" + BACKEND + "_lora"))
dora_model.save_pretrained(ARTIFACT_ROOT / ("pin_" + BACKEND + "_dora"))
